In [0]:
%pip install -U --quiet langchain langchain-community langchainhub langchain-openai langchain-databricks databricks-langchain langgraph mlflow

In [0]:
# Databricks notebook source
# foodly_rag_agent_notebook.py
# Full pipeline: Ingest → Chunk → Delta → Vector Index → LangGraph Agent → MLflow → UC Registry → Serving Endpoint

# COMMAND ----------
# MAGIC %md
# MAGIC # 🍔 Foodly RAG Agent — Full Pipeline
# MAGIC ### Phases:
# MAGIC - **Phase 0** — Install dependencies
# MAGIC - **Phase 1** — Load, chunk & save to Delta
# MAGIC - **Phase 2** — Create Vector Search Index
# MAGIC - **Phase 3** — Build LangGraph ReAct Agent
# MAGIC - **Phase 4** — MLflow logging
# MAGIC - **Phase 5** — Register model in Unity Catalog
# MAGIC - **Phase 6** — Create Serving Endpoint
# MAGIC - **Phase 7** — Test the live endpoint

# COMMAND ----------
# MAGIC %md
# MAGIC ## ⚙️ Phase 0 — Install Dependencies

# COMMAND ----------

# MAGIC %pip install -U --quiet \
# MAGIC     langchain langchain-community langchainhub \
# MAGIC     langchain-openai langchain-databricks \
# MAGIC     databricks-langchain langgraph mlflow

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🔧 Global Config — Update These Before Running

# COMMAND ----------

# ── User config ────────────────────────────────────────────────────────────────
DATABRICKS_HOST        = "https://dbc-921bc807-9d50.cloud.databricks.com/"
SECRET_SCOPE           = ""
SECRET_KEY             = "databricks_pat"

CATALOG                = "dev_agents"
SCHEMA                 = "naval"
TABLE_NAME             = f"{CATALOG}.{SCHEMA}.foodly_docs"
UC_MODEL_NAME          = f"{CATALOG}.{SCHEMA}.foodly_rag_agent"
VECTOR_INDEX_NAME      = f"{CATALOG}.{SCHEMA}.foodly_index"
VS_ENDPOINT_NAME       = "vs_mindsprint"   # existing VS endpoint name
EMBEDDING_ENDPOINT     = "databricks-gte-large-en"        # embedding model endpoint
LLM_ENDPOINT           = "databricks-gpt-oss-120b"
SERVING_ENDPOINT_NAME  = "foodly-rag-agent-endpoint"
MLFLOW_EXPERIMENT_PATH = "/Users/tlyemul@gmail.com/foodly_rag_agent"

FILE_PATH = "/Volumes/dev_agents/naval/raw/foodly/foodly_company_documents (1).txt"
# ───────────────────────────────────────────────────────────────────────────────

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📄 Phase 1 — Load, Chunk & Save to Delta

# COMMAND ----------

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Load raw document ──────────────────────────────────────────────────────────
loader    = TextLoader(FILE_PATH, encoding="utf-8")
documents = loader.load()
print(f"✅ Loaded {len(documents)} document(s).")

# ── Chunk ──────────────────────────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
docs = text_splitter.split_documents(documents)
print(f"✅ Created {len(docs)} chunks.")

# ── Build chunk dataframe ──────────────────────────────────────────────────────
chunk_data = [
    {
        "chunk_id": i + 1,
        "content":  d.page_content,
        "metadata": str(d.metadata)
    }
    for i, d in enumerate(docs)
]

df = spark.createDataFrame(chunk_data)
df.display()

# ── Save to Delta ──────────────────────────────────────────────────────────────
df.write.mode("overwrite").saveAsTable(TABLE_NAME)
print(f"✅ Saved to Delta table: {TABLE_NAME}")

# ── Enable Change Data Feed (required for Vector Search sync) ──────────────────
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("✅ Change Data Feed enabled.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🔍 Phase 2 — Create Vector Search Index

# COMMAND ----------

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Check if index already exists; skip creation if so
existing_indexes = [idx["name"] for idx in vsc.list_indexes(VS_ENDPOINT_NAME).get("vector_indexes", [])]

if VECTOR_INDEX_NAME in existing_indexes:
    print(f"ℹ️  Index '{VECTOR_INDEX_NAME}' already exists — skipping creation.")
else:
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=VECTOR_INDEX_NAME,
        source_table_name=TABLE_NAME,
        pipeline_type="TRIGGERED",           # use CONTINUOUS for real-time sync
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT
    )
    print(f"✅ Vector Search index created: {VECTOR_INDEX_NAME}")

# ── Trigger a sync to pick up latest data ─────────────────────────────────────
vsc.get_index(VS_ENDPOINT_NAME, VECTOR_INDEX_NAME).sync()
print("✅ Index sync triggered.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🤖 Phase 3 — Build LangGraph ReAct Agent

# COMMAND ----------

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START
from langgraph.graph import MessagesState
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

# ── Vector Search Retriever Tool ───────────────────────────────────────────────
vs_tool = VectorSearchRetrieverTool(
    index_name=VECTOR_INDEX_NAME,
    tool_name="foodly_policy_document_retrieval_tool",
    num_results=4,
    tool_description=(
        "Use this tool to search the Foodly knowledge base for policies, procedures, "
        "and service-related information. It retrieves the most relevant chunks from "
        "the company's official documentation, including refund rules, cancellation terms, "
        "delivery guidelines, loyalty program details, privacy policies, and escalation procedures."
    )
)

llm_with_tools = llm.bind_tools([vs_tool])

# ── Agent node ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = SystemMessage(content="""You are a helpful Foodly customer support assistant.
Always use the retrieval tool to answer questions about Foodly policies and procedures.
Do not answer from memory or make up information.
If the information is not found in the knowledge base, clearly say so.""")

def call_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])]}

# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(MessagesState)
builder.add_node("llm",   call_llm)
builder.add_node("tools", ToolNode([vs_tool]))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

agent = builder.compile()
print("✅ LangGraph agent compiled.")

# ── Quick smoke test ───────────────────────────────────────────────────────────
try:
    test_result   = agent.invoke({"messages": [HumanMessage("What are refund policies for Foodly?")]})
    sample_output = test_result["messages"][-1].content
    print(f"\n📋 Sample output:\n{sample_output}")
except Exception as e:
    print(f"❌ Agent test failed: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📊 Phase 4 — MLflow Logging

# COMMAND ----------

import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)

with mlflow.start_run(run_name="foodly_rag_agent_v1") as run:

    # ── Log parameters ─────────────────────────────────────────────────────────
    mlflow.log_param("llm_endpoint",      LLM_ENDPOINT)
    mlflow.log_param("embedding_endpoint", EMBEDDING_ENDPOINT)
    mlflow.log_param("vector_index",      VECTOR_INDEX_NAME)
    mlflow.log_param("chunk_size",        800)
    mlflow.log_param("chunk_overlap",     150)
    mlflow.log_param("num_results",       4)
    mlflow.log_param("total_chunks",      len(docs))

    # ── Log metrics ────────────────────────────────────────────────────────────
    mlflow.log_metric("num_messages_in_trace", len(test_result["messages"]))

    # ── Log sample output as artifact ──────────────────────────────────────────
    with open("/tmp/sample_output.txt", "w") as f:
        f.write(sample_output)
    mlflow.log_artifact("/tmp/sample_output.txt")

    RUN_ID = run.info.run_id
    print(f"✅ MLflow run logged. Run ID: {RUN_ID}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📦 Phase 5 — Register Model in Unity Catalog

# COMMAND ----------

import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# ── Define model signature ─────────────────────────────────────────────────────
input_schema  = Schema([ColSpec("string", "query")])
output_schema = Schema([ColSpec("string", "response")])
signature     = ModelSignature(inputs=input_schema, outputs=output_schema)

# ── PythonModel wrapper ────────────────────────────────────────────────────────
class FoodlyAgentWrapper(mlflow.pyfunc.PythonModel):
    """
    Wraps the LangGraph ReAct agent as an MLflow PythonModel.
    Rebuilds the agent on load so it works in the serving container.
    """

    def load_context(self, context):
        from langchain_core.messages import HumanMessage, SystemMessage
        from langgraph.graph import StateGraph, START
        from langgraph.graph import MessagesState
        from langgraph.prebuilt.tool_node import ToolNode, tools_condition
        from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

        _llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

        _vs_tool = VectorSearchRetrieverTool(
            index_name=VECTOR_INDEX_NAME,
            tool_name="foodly_policy_document_retrieval_tool",
            num_results=4,
            tool_description=(
                "Use this tool to search the Foodly knowledge base for policies, "
                "procedures, and service-related information."
            )
        )

        _llm_with_tools = _llm.bind_tools([_vs_tool])

        _system = SystemMessage(content="""You are a helpful Foodly customer support assistant.
Always use the retrieval tool to answer questions about Foodly policies.
Do not answer from memory. If information is not found, say so clearly.""")

        def _call_llm(state: MessagesState):
            return {"messages": [_llm_with_tools.invoke([_system] + state["messages"])]}

        _builder = StateGraph(MessagesState)
        _builder.add_node("llm",   _call_llm)
        _builder.add_node("tools", ToolNode([_vs_tool]))
        _builder.add_edge(START, "llm")
        _builder.add_conditional_edges("llm", tools_condition)
        _builder.add_edge("tools", "llm")

        self.agent = _builder.compile()

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        results = []
        for _, row in model_input.iterrows():
            query = row["query"]
            try:
                response = self.agent.invoke({"messages": [HumanMessage(query)]})
                answer   = response["messages"][-1].content
            except Exception as e:
                answer = f"Error: {str(e)}"
            results.append(answer)
        return pd.Series(results)


# ── Log & register to UC ───────────────────────────────────────────────────────
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="foodly_rag_agent_register") as run:
    mlflow.pyfunc.log_model(
        artifact_path="foodly_agent",
        python_model=FoodlyAgentWrapper(),
        signature=signature,
        registered_model_name=UC_MODEL_NAME,
        pip_requirements=[
            "langchain",
            "langchain-community",
            "langchain-databricks",
            "databricks-langchain",
            "langgraph",
            "mlflow"
        ]
    )
    REGISTER_RUN_ID = run.info.run_id
    print(f"✅ Model registered in UC: {UC_MODEL_NAME}")
    print(f"   Run ID: {REGISTER_RUN_ID}")

# ── Get the latest registered version ─────────────────────────────────────────
client         = mlflow.tracking.MlflowClient()
latest_version = client.get_registered_model(UC_MODEL_NAME).latest_versions[-1].version
print(f"✅ Latest model version: {latest_version}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🚀 Phase 6 — Create Serving Endpoint

# COMMAND ----------

import requests, json

DATABRICKS_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

endpoint_config = {
    "name": SERVING_ENDPOINT_NAME,
    "config": {
        "served_models": [
            {
                "name":                  "foodly-agent-v1",
                "model_name":            UC_MODEL_NAME,
                "model_version":         str(latest_version),
                "workload_size":         "Small",   # Small / Medium / Large
                "scale_to_zero_enabled": True        # cost-saving for dev/test
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name":  "foodly-agent-v1",
                    "traffic_percentage": 100
                }
            ]
        }
    }
}

response = requests.post(
    url=f"{DATABRICKS_HOST}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
    json=endpoint_config
)

if response.status_code == 200:
    print(f"✅ Serving endpoint creation initiated: {SERVING_ENDPOINT_NAME}")
    print(json.dumps(response.json(), indent=2))
else:
    print(f"❌ Failed to create endpoint: {response.status_code}")
    print(response.text)

# COMMAND ----------
# MAGIC %md
# MAGIC ## ⏳ Phase 6b — Poll Until Endpoint is Ready (Optional)

# COMMAND ----------

import time

def wait_for_endpoint(host, token, endpoint_name, timeout_minutes=20):
    """Polls the endpoint status until it becomes READY or times out."""
    url     = f"{host}/api/2.0/serving-endpoints/{endpoint_name}"
    headers = {"Authorization": f"Bearer {token}"}
    timeout = timeout_minutes * 60
    elapsed = 0

    print(f"⏳ Waiting for endpoint '{endpoint_name}' to be ready...")
    while elapsed < timeout:
        resp   = requests.get(url, headers=headers)
        state  = resp.json().get("state", {}).get("ready", "NOT_READY")
        config = resp.json().get("state", {}).get("config_update", "")
        print(f"   Status: {state} | Config: {config} | Elapsed: {elapsed}s")

        if state == "READY":
            print(f"\n✅ Endpoint is READY: {endpoint_name}")
            return True

        time.sleep(30)
        elapsed += 30

    print(f"❌ Timed out after {timeout_minutes} minutes.")
    return False

wait_for_endpoint(DATABRICKS_HOST, DATABRICKS_TOKEN, SERVING_ENDPOINT_NAME)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🧪 Phase 7 — Test the Live Endpoint

# COMMAND ----------

def query_endpoint(question: str) -> str:
    """Send a query to the deployed serving endpoint and return the response."""
    response = requests.post(
        url=f"{DATABRICKS_HOST}/serving-endpoints/{SERVING_ENDPOINT_NAME}/invocations",
        headers={
            "Authorization":  f"Bearer {DATABRICKS_TOKEN}",
            "Content-Type":   "application/json"
        },
        json={"dataframe_records": [{"query": question}]}
    )
    if response.status_code == 200:
        return response.json()["predictions"][0]
    else:
        return f"❌ Error {response.status_code}: {response.text}"


# ── Run test queries ───────────────────────────────────────────────────────────
test_questions = [
    "What are the refund policies for Foodly?",
    "How does the Foodly loyalty program work?",
    "What is the cancellation policy for orders?",
    "How do I escalate a complaint?"
]

print("=" * 70)
for question in test_questions:
    print(f"\n❓ Question: {question}")
    answer = query_endpoint(question)
    print(f"💬 Answer:   {answer}")
    print("-" * 70)

In [0]:
# Databricks notebook source
# foodly_rag_agent_notebook_v2.py
# Full pipeline: Ingest → Chunk → Delta → Vector Index → LangGraph Agent → MLflow → UC Registry → Serving Endpoint
# v2: Fixed cloudpickle/Pydantic serialization error using code_paths approach

# COMMAND ----------
# MAGIC %md
# MAGIC # 🍔 Foodly RAG Agent — Full Pipeline (v2)
# MAGIC ### Phases:
# MAGIC - **Phase 0** — Install dependencies
# MAGIC - **Phase 1** — Global config
# MAGIC - **Phase 2** — Load, chunk & save to Delta
# MAGIC - **Phase 3** — Create Vector Search Index
# MAGIC - **Phase 4** — Build & smoke-test LangGraph ReAct Agent
# MAGIC - **Phase 5** — MLflow logging
# MAGIC - **Phase 6** — Register model in Unity Catalog *(with Pydantic fix)*
# MAGIC - **Phase 7** — Create Serving Endpoint
# MAGIC - **Phase 8** — Poll until endpoint is ready
# MAGIC - **Phase 9** — Test the live endpoint

# COMMAND ----------
# MAGIC %md
# MAGIC ## ⚙️ Phase 0 — Install Dependencies

# COMMAND ----------

# MAGIC %pip install -U --quiet \
# MAGIC     langchain>=0.2 \
# MAGIC     langchain-community>=0.2 \
# MAGIC     langchainhub \
# MAGIC     langchain-openai \
# MAGIC     langchain-databricks>=0.1 \
# MAGIC     databricks-langchain>=0.1 \
# MAGIC     langgraph>=0.1 \
# MAGIC     mlflow>=2.12 \
# MAGIC     pydantic>=2.0,\<3.0

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🔧 Phase 1 — Global Config
# MAGIC **Update all values in this cell before running the notebook.**

# COMMAND ----------


# ── Workspace ──────────────────────────────────────────────────────────────────
DATABRICKS_HOST        = "https://dbc-921bc807-9d50.cloud.databricks.com/"
SECRET_SCOPE           = ""
SECRET_KEY             = "databricks_pat"

# ── Unity Catalog ──────────────────────────────────────────────────────────────
CATALOG                = "dev_agents"
SCHEMA                 = "naval"
TABLE_NAME             = f"{CATALOG}.{SCHEMA}.foodly_docs"
UC_MODEL_NAME          = f"{CATALOG}.{SCHEMA}.foodly_rag_agent"
VECTOR_INDEX_NAME      = f"{CATALOG}.{SCHEMA}.foodly_index"

# ── Databricks endpoints ───────────────────────────────────────────────────────
VS_ENDPOINT_NAME       = "vs_mindsprint"    # existing VS endpoint
EMBEDDING_ENDPOINT     = "databricks-gte-large-en"        # embedding model
LLM_ENDPOINT           = "databricks-gpt-oss-120b"        # chat model
SERVING_ENDPOINT_NAME  = "foodly-rag-agent-endpoint"      # new serving endpoint

# ── MLflow ─────────────────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT_PATH = "/Users/tlyemul@gmail.com/foodly_rag_agent"

# ── Source data ────────────────────────────────────────────────────────────────
FILE_PATH              = "/Volumes/dev_agents/naval/raw/foodly/foodly_company_documents (1).txt"

# ── Local paths (driver node) ──────────────────────────────────────────────────
WRAPPER_PATH           = "/tmp/foodly_agent_wrapper.py"

print("✅ Config loaded.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📄 Phase 2 — Load, Chunk & Save to Delta

# COMMAND ----------

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Load raw document ──────────────────────────────────────────────────────────
loader    = TextLoader(FILE_PATH, encoding="utf-8")
documents = loader.load()
print(f"✅ Loaded {len(documents)} document(s).")
print(f"   Metadata: {documents[0].metadata}")

# ── Chunk ──────────────────────────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
docs = text_splitter.split_documents(documents)
print(f"✅ Created {len(docs)} chunks.")

# ── Build rows ─────────────────────────────────────────────────────────────────
chunk_data = [
    {
        "chunk_id": i + 1,
        "content":  d.page_content,
        "metadata": str(d.metadata)
    }
    for i, d in enumerate(docs)
]

df = spark.createDataFrame(chunk_data)
df.display()

# ── Save to Delta ──────────────────────────────────────────────────────────────
df.write.mode("overwrite").saveAsTable(TABLE_NAME)
print(f"✅ Saved to Delta table: {TABLE_NAME}")

# ── Enable Change Data Feed (required for Vector Search sync) ──────────────────
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("✅ Change Data Feed enabled.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🔍 Phase 3 — Create Vector Search Index

# COMMAND ----------

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# ── Create index only if it doesn't exist ─────────────────────────────────────
existing = [
    idx["name"]
    for idx in vsc.list_indexes(VS_ENDPOINT_NAME).get("vector_indexes", [])
]

if VECTOR_INDEX_NAME in existing:
    print(f"ℹ️  Index '{VECTOR_INDEX_NAME}' already exists — skipping creation.")
else:
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=VECTOR_INDEX_NAME,
        source_table_name=TABLE_NAME,
        pipeline_type="TRIGGERED",           # CONTINUOUS for real-time sync
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT
    )
    print(f"✅ Vector Search index created: {VECTOR_INDEX_NAME}")

# ── Sync to pick up latest data ────────────────────────────────────────────────
vsc.get_index(VS_ENDPOINT_NAME, VECTOR_INDEX_NAME).sync()
print("✅ Index sync triggered.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🤖 Phase 4 — Build & Smoke-Test LangGraph ReAct Agent

# COMMAND ----------

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START
from langgraph.graph import MessagesState
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

# ── Retriever tool ─────────────────────────────────────────────────────────────
vs_tool = VectorSearchRetrieverTool(
    index_name=VECTOR_INDEX_NAME,
    tool_name="foodly_policy_document_retrieval_tool",
    num_results=4,
    tool_description=(
        "Use this tool to search the Foodly knowledge base for policies, procedures, "
        "and service-related information. It retrieves the most relevant chunks from "
        "the company's official documentation, including refund rules, cancellation terms, "
        "delivery guidelines, loyalty program details, privacy policies, and escalation procedures."
    )
)

llm_with_tools = llm.bind_tools([vs_tool])

# ── System prompt ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = SystemMessage(content="""You are a helpful Foodly customer support assistant.
Always use the retrieval tool to answer questions about Foodly policies and procedures.
Do not answer from memory or make up information.
If the information is not found in the knowledge base, clearly say so.""")

# ── Agent node ─────────────────────────────────────────────────────────────────
def call_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])]}

# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(MessagesState)
builder.add_node("llm",   call_llm)
builder.add_node("tools", ToolNode([vs_tool]))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

agent = builder.compile()
print("✅ LangGraph agent compiled.")

# ── Smoke test ─────────────────────────────────────────────────────────────────
try:
    test_result   = agent.invoke({"messages": [HumanMessage("What are refund policies for Foodly?")]})
    sample_output = test_result["messages"][-1].content
    print(f"\n📋 Smoke test output:\n{sample_output}")
except Exception as e:
    print(f"❌ Agent smoke test failed: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📊 Phase 5 — MLflow Logging

# COMMAND ----------

import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)

with mlflow.start_run(run_name="foodly_rag_agent_v2") as run:

    # Parameters
    mlflow.log_param("llm_endpoint",       LLM_ENDPOINT)
    mlflow.log_param("embedding_endpoint", EMBEDDING_ENDPOINT)
    mlflow.log_param("vector_index",       VECTOR_INDEX_NAME)
    mlflow.log_param("chunk_size",         800)
    mlflow.log_param("chunk_overlap",      150)
    mlflow.log_param("num_results",        4)
    mlflow.log_param("total_chunks",       len(docs))

    # Metrics
    mlflow.log_metric("num_messages_in_trace", len(test_result["messages"]))

    # Sample output artifact
    with open("/tmp/sample_output.txt", "w") as f:
        f.write(sample_output)
    mlflow.log_artifact("/tmp/sample_output.txt")

    RUN_ID = run.info.run_id
    print(f"✅ MLflow run logged.")
    print(f"   Run ID : {RUN_ID}")
    print(f"   Experiment: {MLFLOW_EXPERIMENT_PATH}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📦 Phase 6 — Register Model in Unity Catalog
# MAGIC ### ⚡ Key fix: `code_paths` instead of cloudpickle
# MAGIC
# MAGIC **Why the serving endpoint was failing:**
# MAGIC When using `python_model=instance`, MLflow serializes the entire object
# MAGIC (including all LangChain/Pydantic internals) via `cloudpickle`.
# MAGIC The serving container runs a different Pydantic version and fails to
# MAGIC deserialize it → `AttributeError: 'FieldInfo' object has no attribute 'evaluated'`
# MAGIC
# MAGIC **The fix:** Write the model class to a `.py` file and pass it via
# MAGIC `code_paths`. MLflow ships the raw Python source and imports it fresh
# MAGIC at serve time — no pickle, no Pydantic conflict.

# COMMAND ----------
# MAGIC %md
# MAGIC ### Step 6a — Write wrapper class to a Python file

# COMMAND ----------

wrapper_code = f'''
import mlflow
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START
from langgraph.graph import MessagesState
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

# ── Config (hardcoded so the file is self-contained) ──────────────────────────
LLM_ENDPOINT      = "{LLM_ENDPOINT}"
VECTOR_INDEX_NAME = "{VECTOR_INDEX_NAME}"
SYSTEM_CONTENT    = """You are a helpful Foodly customer support assistant.
Always use the retrieval tool to answer questions about Foodly policies.
Do not answer from memory. If information is not found, say so clearly."""


class FoodlyAgentWrapper(mlflow.pyfunc.PythonModel):
    """
    MLflow PythonModel wrapper for the Foodly LangGraph RAG agent.
    Uses code_paths to avoid cloudpickle / Pydantic version conflicts.
    """

    def load_context(self, context):
        """Rebuild the agent fresh at serve time."""
        _llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

        _vs_tool = VectorSearchRetrieverTool(
            index_name=VECTOR_INDEX_NAME,
            tool_name="foodly_policy_document_retrieval_tool",
            num_results=4,
            tool_description=(
                "Use this tool to search the Foodly knowledge base for policies, "
                "procedures, and service-related information."
            )
        )

        _llm_with_tools = _llm.bind_tools([_vs_tool])
        _system = SystemMessage(content=SYSTEM_CONTENT)

        def _call_llm(state: MessagesState):
            return {{"messages": [_llm_with_tools.invoke([_system] + state["messages"])]}}

        _builder = StateGraph(MessagesState)
        _builder.add_node("llm",   _call_llm)
        _builder.add_node("tools", ToolNode([_vs_tool]))
        _builder.add_edge(START, "llm")
        _builder.add_conditional_edges("llm", tools_condition)
        _builder.add_edge("tools", "llm")

        self.agent = _builder.compile()

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        """Called by the serving endpoint for each batch of requests."""
        results = []
        for _, row in model_input.iterrows():
            query = row["query"]
            try:
                response = self.agent.invoke({{"messages": [HumanMessage(query)]}})
                answer   = response["messages"][-1].content
            except Exception as e:
                answer = f"Error: {{str(e)}}"
            results.append(answer)
        return pd.Series(results)
# MLflow calls this function directly at serve time — NO cloudpickle involved
def _load_pyfunc(path):
    """Entry point for MLflow loader_module. Instantiates model fresh."""
    model = FoodlyAgentWrapper()
    return model

'''

with open(WRAPPER_PATH, "w") as f:
    f.write(wrapper_code)

print(f"✅ Wrapper written to: {WRAPPER_PATH}")

# COMMAND ----------
# MAGIC %md
# MAGIC ### Step 6b — Log & register model using code_paths

# COMMAND ----------
# Phase 6b — Register model using loader_module (completely bypasses cloudpickle)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 📦 Phase 6 — Register Model using mlflow.langchain (Definitive Fix)
# MAGIC Uses LangChain-native serialization — no cloudpickle, no Pydantic conflicts

# COMMAND ----------

import mlflow
import mlflow.langchain
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

mlflow.set_registry_uri("databricks-uc")

# ── Model signature ────────────────────────────────────────────────────────────
input_schema  = Schema([ColSpec("string", "query")])
output_schema = Schema([ColSpec("string", "response")])
signature     = ModelSignature(inputs=input_schema, outputs=output_schema)

# ── Log with LangChain flavor ──────────────────────────────────────────────────
with mlflow.start_run(run_name="foodly_rag_agent_register_v4") as run:
    mlflow.langchain.log_model(
        lc_model=agent,                        # your compiled LangGraph agent
        artifact_path="foodly_agent",
        signature=signature,
        registered_model_name=UC_MODEL_NAME,
        pip_requirements=[
            "pydantic>=2.0,<3.0",
            "langchain>=0.2",
            "langchain-community>=0.2",
            "langchain-databricks>=0.1",
            "databricks-langchain>=0.1",
            "langgraph>=0.1",
            "mlflow>=2.12"
        ]
    )
    print(f"✅ Model registered in UC : {UC_MODEL_NAME}")
    print(f"   Run ID                 : {run.info.run_id}")

# ── Get latest version ─────────────────────────────────────────────────────────
client         = mlflow.tracking.MlflowClient()
latest_version = client.get_registered_model(UC_MODEL_NAME).latest_versions[-1].version
print(f"✅ Latest model version: {latest_version}")
# COMMAND ----------
# MAGIC %md
# MAGIC ## 🚀 Phase 7 — Create or Update Serving Endpoint

# COMMAND ----------

import requests, json

DATABRICKS_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
HEADERS          = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type":  "application/json"
}

# ── Check if endpoint already exists ──────────────────────────────────────────
check = requests.get(
    url=f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/{SERVING_ENDPOINT_NAME}",
    headers=HEADERS
)
endpoint_exists = check.status_code == 200

# ── Served model config ────────────────────────────────────────────────────────
served_model_config = {
    "served_models": [
        {
            "name":                  "foodly-agent-v2",
            "model_name":            UC_MODEL_NAME,
            "model_version":         str(latest_version),
            "workload_size":         "Small",    # Small / Medium / Large
            "scale_to_zero_enabled": True         # cost-saving for dev/test
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name":  "foodly-agent-v2",
                "traffic_percentage": 100
            }
        ]
    }
}

if endpoint_exists:
    # ── UPDATE existing endpoint ───────────────────────────────────────────────
    print(f"ℹ️  Endpoint '{SERVING_ENDPOINT_NAME}' exists — updating to version {latest_version}...")
    response = requests.put(
        url=f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/{SERVING_ENDPOINT_NAME}/config",
        headers=HEADERS,
        json=served_model_config
    )
else:
    # ── CREATE new endpoint ────────────────────────────────────────────────────
    print(f"ℹ️  Creating new endpoint '{SERVING_ENDPOINT_NAME}'...")
    response = requests.post(
        url=f"{DATABRICKS_HOST}/api/2.0/serving-endpoints",
        headers=HEADERS,
        json={"name": SERVING_ENDPOINT_NAME, "config": served_model_config}
    )

if response.status_code in (200, 201):
    print(f"✅ Serving endpoint {'updated' if endpoint_exists else 'created'} successfully!")
    print(json.dumps(response.json(), indent=2))
else:
    print(f"❌ Failed: {response.status_code}")
    print(response.text)

# COMMAND ----------
# MAGIC %md
# MAGIC ## ⏳ Phase 8 — Poll Until Endpoint is Ready

# COMMAND ----------

import time

def wait_for_endpoint(host, headers, endpoint_name, timeout_minutes=25):
    """Polls every 30s until the endpoint reaches READY state or times out."""
    url     = f"{host}/api/2.0/serving-endpoints/{endpoint_name}"
    timeout = timeout_minutes * 60
    elapsed = 0

    print(f"⏳ Waiting for endpoint '{endpoint_name}' to be READY...")
    print(f"   Timeout: {timeout_minutes} minutes\n")

    while elapsed < timeout:
        resp  = requests.get(url, headers=headers)
        body  = resp.json()
        state = body.get("state", {}).get("ready", "NOT_READY")
        cfg   = body.get("state", {}).get("config_update", "—")

        print(f"   [{elapsed:>4}s]  Status: {state:<12}  Config update: {cfg}")

        if state == "READY":
            print(f"\n✅ Endpoint is READY: {endpoint_name}")
            return True

        if state == "FAILED":
            print(f"\n❌ Endpoint entered FAILED state. Check the Serving UI for details.")
            return False

        time.sleep(30)
        elapsed += 30

    print(f"\n❌ Timed out after {timeout_minutes} minutes.")
    return False

wait_for_endpoint(DATABRICKS_HOST, HEADERS, SERVING_ENDPOINT_NAME)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 🧪 Phase 9 — Test the Live Endpoint

# COMMAND ----------

def query_endpoint(question: str) -> str:
    """Send a single query to the deployed serving endpoint."""
    response = requests.post(
        url=f"{DATABRICKS_HOST}/serving-endpoints/{SERVING_ENDPOINT_NAME}/invocations",
        headers=HEADERS,
        json={"dataframe_records": [{"query": question}]}
    )
    if response.status_code == 200:
        return response.json()["predictions"][0]
    else:
        return f"❌ Error {response.status_code}: {response.text}"


# ── Run test queries ───────────────────────────────────────────────────────────
test_questions = [
    "What are the refund policies for Foodly?",
    "How does the Foodly loyalty program work?",
    "What is the cancellation policy for orders?",
    "How do I escalate a complaint?",
    "What are the delivery guidelines?"
]

print("=" * 70)
print(f"  Endpoint : {SERVING_ENDPOINT_NAME}")
print(f"  Model    : {UC_MODEL_NAME}  v{latest_version}")
print("=" * 70)

for question in test_questions:
    print(f"\n❓ {question}")
    answer = query_endpoint(question)
    print(f"💬 {answer}")
    print("-" * 70)

In [0]:
# COMMAND ----------
# Phase 9 — Test endpoint (updated for langchain flavor input format)

def query_endpoint(question: str) -> str:
    response = requests.post(
        url=f"{DATABRICKS_HOST}/serving-endpoints/{SERVING_ENDPOINT_NAME}/invocations",
        headers=HEADERS,
        json={
            "inputs": [{"query": question}]     # ← langchain flavor uses "inputs"
        }
    )
    if response.status_code == 200:
        return response.json().get("predictions", [response.json()])[0]
    else:
        return f"❌ Error {response.status_code}: {response.text}"

In [0]:
sales data docs, excel

data preprocessing (chunking )
embedding
vector database
retriver
Agent
model version
registry to UC 
serving endpoint


App